# 04A_v3 — Extended Exact Pre-Challenge Fire-Approach Reconstruction

**Build ID: `04A_V3_EXTENDED_EXACT_APPROACH`**

This is a focused extension of 04A_v2.

### Frozen and unchanged
- `FROZEN_V1` Held/Burned Over target
- corrected local fireline units
- operational prediction anchor: earliest supported-event start
- local incoming-fire direction methodology

### Why V3 exists
The 03L trajectory cache used `251 m` as a sentinel for distances beyond the 250 m retained neighborhood. That was correct for challenge detection, but it artificially creates many zero approach rates.

V3 therefore reconstructs **uncapped exact vector boundary-distance magnitudes** at:

\[
t_{-3}, t_{-2}, t_{-1}, t_0
\]

using the existing NIROPS polygons and existing 03L sample points.

No external environmental data are downloaded and no NIROPS raster is rebuilt.

The old 03L signed-distance cache is reused only for the already-established inside/outside diagnostic; its capped magnitude is not used for the new exact approach-rate calculation.

In [ ]:
import os  # Configurable project root for the public release.
NOTEBOOK_BUILD_ID = "04A_V3_EXTENDED_EXACT_APPROACH"

from pathlib import Path
import sys, subprocess, importlib.util

PROJECT_ROOT = Path(os.environ.get("FIRELINE_PROJECT_ROOT", "."))

CORRECTED_ROOT = PROJECT_ROOT / "data" / "processed" / "fireline_corrected_local_250m_10m"
FROZEN_ROOT = CORRECTED_ROOT / "frozen_cohort"

DYNAMIC_PATH = FROZEN_ROOT / "fireline_ml_dynamic_timing_cohort_frozen_v1.parquet"

APPROACH_V2_ROOT = PROJECT_ROOT / "data" / "processed" / "fireline_dynamic_approach_v2"
APPROACH_V2_PATH = APPROACH_V2_ROOT / "dynamic_approach_features_v2.parquet"

UNIT_ROOT = CORRECTED_ROOT / "units"
SAMPLE_ROOT = CORRECTED_ROOT / "sample_points"
TRAJECTORY_ROOT = CORRECTED_ROOT / "trajectories"

NIROPS_SOURCE = Path(
    (os.environ.get("FIRELINE_PROJECT_ROOT", ".") + '/data/interim/joinability_audit/nirops_extracted/NIROPS_2020_2024_R1_R6/NIROPS_2020_2024_R1_R6.shp')
)

PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / "fireline_extended_exact_approach_v3"
SHARD_ROOT = PROCESSED_ROOT / "incident_shards"
NIROPS_CACHE_ROOT = PROCESSED_ROOT / "nirops_progression_cache"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "extended_exact_approach_v3"

for p in [PROCESSED_ROOT, SHARD_ROOT, NIROPS_CACHE_ROOT, OUTPUT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

for p in [
    DYNAMIC_PATH, APPROACH_V2_PATH, UNIT_ROOT, SAMPLE_ROOT,
    TRAJECTORY_ROOT, NIROPS_SOURCE
]:
    assert p.exists(), p

CRS_METRIC = "EPSG:5070"
MAX_HISTORY_OBSERVATIONS = 4
ETA_CAP_18H = 18.0
ETA_CAP_48H = 48.0
RESUME = True

print("=" * 92)
print("RUNNING:", NOTEBOOK_BUILD_ID)
print("=" * 92)
print("04A_v2:", APPROACH_V2_PATH)
print("NIROPS:", NIROPS_SOURCE)
print("V3 output:", PROCESSED_ROOT)

In [ ]:
required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "geopandas": "geopandas",
    "pyarrow": "pyarrow",
    "shapely": "shapely",
    "tqdm": "tqdm",
    "pyogrio": "pyogrio",
}

for module, package in required.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import gc, math, time, traceback
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 500)
pd.set_option("display.width", 280)

print("Python:", sys.executable)
print("pandas:", pd.__version__)
print("GeoPandas:", gpd.__version__)
print("Shapely:", shapely.__version__)

if int(shapely.__version__.split(".")[0]) < 2:
    raise RuntimeError("04A_v3 requires Shapely >= 2.")

## 1. Core readers and normalization

In [ ]:
def norm_irwin(series):
    return (
        series.astype("string")
        .str.strip()
        .str.replace("{", "", regex=False)
        .str.replace("}", "", regex=False)
        .str.replace("-", "", regex=False)
        .str.lower()
        .replace({
            "<na>": pd.NA,
            "nan": pd.NA,
            "none": pd.NA,
            "": pd.NA,
        })
    )


def read_03l_units(path):
    df = pd.read_parquet(path)

    if "geometry" not in df.columns:
        raise RuntimeError(f"No geometry column in {path}")

    nonnull = df["geometry"].dropna()

    if not len(nonnull):
        raise RuntimeError(f"Empty geometry column in {path}")

    example = nonnull.iloc[0]

    if not isinstance(example, (bytes, bytearray, memoryview)):
        raise RuntimeError(
            f"Expected WKB bytes, got {type(example)} in {path}"
        )

    geometry = gpd.GeoSeries.from_wkb(
        df["geometry"],
        crs=CRS_METRIC,
    )

    return gpd.GeoDataFrame(
        df.drop(columns=["geometry"]),
        geometry=geometry,
        crs=CRS_METRIC,
    )


def resolve_shard(root, raw_irwin, suffix):
    direct = root / f"{raw_irwin}{suffix}"

    if direct.exists():
        return direct

    target = norm_irwin(pd.Series([raw_irwin])).iloc[0]

    matches = []

    for path in root.glob(f"*{suffix}"):
        candidate = norm_irwin(pd.Series([path.stem])).iloc[0]

        if candidate == target:
            matches.append(path)

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not uniquely resolve {raw_irwin} under {root}; matches={len(matches)}"
        )

    return matches[0]

## 2. Load 03N dynamic cohort and accepted 04A_v2 anchor/direction features

In [ ]:
dynamic = pd.read_parquet(DYNAMIC_PATH)
approach_v2 = pd.read_parquet(APPROACH_V2_PATH)

assert dynamic["unit_uid"].is_unique
assert approach_v2["unit_uid"].is_unique
assert len(dynamic) == len(approach_v2)

dynamic["_irwin_04a"] = norm_irwin(dynamic["_irwin_norm"])

required_v2 = [
    "unit_uid",
    "prediction_anchor_idx",
    "prediction_anchor_utc",
    "operational_anchor_clean",
    "prediction_t0_any_inside",
    "local_fire_to_line_distance_geom_m",
    "approach_azimuth_deg",
    "approach_direction_reliable",
    "fireline_orientation_deg",
    "approach_vs_line_angle_deg",
]

missing = [c for c in required_v2 if c not in approach_v2.columns]

if missing:
    raise RuntimeError(f"04A_v2 table missing columns: {missing}")

base = dynamic[
    [
        "unit_uid",
        "_irwin_norm",
        "_irwin_04a",
        "IncidentName",
        "Year",
        "_engagement_norm",
        "hold_label",
    ]
].merge(
    approach_v2[required_v2],
    on="unit_uid",
    how="left",
    validate="one_to_one",
)

print("Dynamic units:", f"{len(base):,}")
print("Incidents:", base["_irwin_norm"].nunique())
print(
    "Clean operational anchors:",
    f"{int(base['operational_anchor_clean'].sum()):,}",
    f"({base['operational_anchor_clean'].mean()*100:.2f}%)",
)

## 3. Load or create projected NIROPS cache

The first run reads the original NIROPS shapefile and writes a GeoParquet containing only the 382 dynamic incidents in EPSG:5070. Later V3 reruns reuse that cache.

In [ ]:
NIROPS_CACHE_PATH = (
    NIROPS_CACHE_ROOT /
    "nirops_dynamic_382_progression_epsg5070.parquet"
)

required_irwins = set(base["_irwin_04a"].dropna())

if NIROPS_CACHE_PATH.exists():
    print("Loading NIROPS V3 cache:")
    print(NIROPS_CACHE_PATH)

    nirops = gpd.read_parquet(NIROPS_CACHE_PATH)

else:
    print("Reading original NIROPS shapefile...")

    g = gpd.read_file(NIROPS_SOURCE)

    cmap = {str(c).lower(): c for c in g.columns}

    irwin_col = cmap.get("_irwin_norm", cmap.get("irwinid"))
    time_col = cmap.get("_time", cmap.get("utc", cmap.get("dateutc")))

    if irwin_col is None or time_col is None:
        raise RuntimeError("NIROPS lacks recognized IRWIN/time columns.")

    g["_irwin_04a"] = norm_irwin(g[irwin_col])

    g["_time_04a"] = pd.to_datetime(
        g[time_col],
        utc=True,
        errors="coerce",
    )

    g = g[
        g["_irwin_04a"].isin(required_irwins)
        & g["_time_04a"].notna()
        & g.geometry.notna()
        & ~g.geometry.is_empty
    ].copy()

    print("Rows retained:", f"{len(g):,}")
    print("Incidents retained:", g["_irwin_04a"].nunique())

    if g.crs is None:
        raise RuntimeError("NIROPS source has no CRS.")

    if str(g.crs).upper() != CRS_METRIC:
        g = g.to_crs(CRS_METRIC)

    dissolved = []

    for irwin, grp in tqdm(
        g.groupby("_irwin_04a", sort=False),
        desc="Union duplicate NIROPS timestamps",
    ):
        d = (
            grp[["_time_04a", "geometry"]]
            .dissolve(by="_time_04a")
            .reset_index()
            .sort_values("_time_04a")
        )

        d["_irwin_04a"] = irwin
        dissolved.append(d)

    nirops = gpd.GeoDataFrame(
        pd.concat(dissolved, ignore_index=True),
        geometry="geometry",
        crs=CRS_METRIC,
    )[["_irwin_04a", "_time_04a", "geometry"]]

    nirops.to_parquet(
        NIROPS_CACHE_PATH,
        index=False,
    )

    del g
    gc.collect()

    print("Saved NIROPS V3 cache:", NIROPS_CACHE_PATH)

missing = required_irwins - set(nirops["_irwin_04a"])

print("NIROPS incidents:", nirops["_irwin_04a"].nunique())
print("Missing required incidents:", len(missing))

if missing:
    display(pd.DataFrame({"missing_irwin": sorted(missing)[:100]}))
    raise RuntimeError("NIROPS V3 cache is incomplete.")

nirops_groups = {
    k: grp.sort_values("_time_04a").reset_index(drop=True)
    for k, grp in nirops.groupby("_irwin_04a", sort=False)
}

## 4. Arc-length sample weighting

In [ ]:
def build_part_length_table(units):
    exploded = (
        units[["unit_row_index", "geometry"]]
        .set_index("unit_row_index")
        .explode(index_parts=True)
        .reset_index()
    )

    if "level_1" in exploded.columns:
        exploded = exploded.rename(columns={"level_1": "part_index"})

    elif "part_index" not in exploded.columns:
        others = [
            c for c in exploded.columns
            if c not in {"unit_row_index", "geometry"}
        ]

        if len(others) != 1:
            raise RuntimeError("Could not identify exploded part index.")

        exploded = exploded.rename(columns={others[0]: "part_index"})

    exploded["part_index"] = exploded["part_index"].astype(np.int16)
    exploded["part_length_m"] = exploded.geometry.length.astype(float)

    return exploded[
        ["unit_row_index", "part_index", "part_length_m"]
    ]


def attach_sample_weights(units, samples):
    part_lengths = build_part_length_table(units)

    counts = (
        samples.groupby(
            ["unit_row_index", "part_index"],
            as_index=False,
        )
        .size()
        .rename(columns={"size": "samples_in_part"})
    )

    result = (
        samples.merge(
            part_lengths,
            on=["unit_row_index", "part_index"],
            how="left",
            validate="many_to_one",
        )
        .merge(
            counts,
            on=["unit_row_index", "part_index"],
            how="left",
            validate="many_to_one",
        )
    )

    if result["part_length_m"].isna().any():
        raise RuntimeError("Sample-to-part weighting failed.")

    result["sample_weight_m"] = (
        result["part_length_m"]
        / result["samples_in_part"]
    )

    return result

## 5. Cached time decoder

In [ ]:
def decode_cached_times(raw_values):
    raw = np.asarray(raw_values, dtype=np.int64)

    valid = raw[raw != np.iinfo(np.int64).min]

    magnitude = float(
        np.median(
            np.abs(valid.astype(np.float64))
        )
    )

    if magnitude >= 1e17:
        unit = "ns"
    elif magnitude >= 1e14:
        unit = "us"
    elif magnitude >= 1e11:
        unit = "ms"
    else:
        unit = "s"

    return (
        pd.DatetimeIndex(
            pd.to_datetime(
                raw,
                unit=unit,
                utc=True,
            )
        ),
        unit,
    )

## 6. Exact vector boundary-distance aggregation

In [ ]:
def weighted_quantile_by_group(
    frame,
    value_col,
    weight_col,
    group_col,
    q,
):
    work = frame[
        [group_col, value_col, weight_col]
    ].copy()

    work = work[
        np.isfinite(work[value_col])
        & np.isfinite(work[weight_col])
        & (work[weight_col] > 0)
    ]

    if not len(work):
        return pd.Series(dtype=float)

    work = work.sort_values(
        [group_col, value_col]
    )

    work["_cum_w"] = (
        work.groupby(group_col)[weight_col]
        .cumsum()
    )

    work["_total_w"] = (
        work.groupby(group_col)[weight_col]
        .transform("sum")
    )

    return (
        work[
            work["_cum_w"]
            >= q * work["_total_w"]
        ]
        .drop_duplicates(group_col, keep="first")
        .set_index(group_col)[value_col]
    )


def summarize_exact_sample_distances(sample_frame):
    x = sample_frame.copy()

    x["_wd"] = (
        x["sample_weight_m"]
        * x["exact_boundary_distance_m"]
    )

    x["_inside_w"] = (
        x["sample_weight_m"]
        * x["cached_inside"].astype(float)
    )

    grouped = (
        x.groupby("unit_uid", sort=False)
        .agg(
            weight_sum=("sample_weight_m", "sum"),
            weighted_distance_sum=("_wd", "sum"),
            exact_min_distance_m=("exact_boundary_distance_m", "min"),
            cached_inside_weight=("_inside_w", "sum"),
            sample_points=("sample_row_index", "size"),
        )
    )

    grouped["exact_mean_distance_m"] = (
        grouped["weighted_distance_sum"]
        / grouped["weight_sum"]
    )

    grouped["cached_inside_fraction"] = (
        grouped["cached_inside_weight"]
        / grouped["weight_sum"]
    )

    grouped["exact_p10_distance_m"] = weighted_quantile_by_group(
        x,
        "exact_boundary_distance_m",
        "sample_weight_m",
        "unit_uid",
        0.10,
    )

    grouped["exact_median_distance_m"] = weighted_quantile_by_group(
        x,
        "exact_boundary_distance_m",
        "sample_weight_m",
        "unit_uid",
        0.50,
    )

    grouped["exact_p90_distance_m"] = weighted_quantile_by_group(
        x,
        "exact_boundary_distance_m",
        "sample_weight_m",
        "unit_uid",
        0.90,
    )

    return (
        grouped[
            [
                "exact_mean_distance_m",
                "exact_median_distance_m",
                "exact_p10_distance_m",
                "exact_p90_distance_m",
                "exact_min_distance_m",
                "cached_inside_fraction",
                "sample_points",
                "weight_sum",
            ]
        ]
        .reset_index()
    )


def exact_distance_for_request_group(
    requested_units,
    weighted_samples,
    signed_cache,
    observation_index,
    fire_geometry,
):
    requested = set(requested_units)

    sample_subset = weighted_samples[
        weighted_samples["unit_uid"].isin(requested)
    ].copy()

    if not len(sample_subset):
        raise RuntimeError("Requested units have no sample rows.")

    points = shapely.points(
        sample_subset["x_m"].to_numpy(float),
        sample_subset["y_m"].to_numpy(float),
    )

    if not fire_geometry.is_valid:
        fire_geometry = shapely.make_valid(fire_geometry)

    boundary = shapely.boundary(fire_geometry)

    distances = np.asarray(
        shapely.distance(points, boundary),
        dtype=float,
    )

    sample_rows = (
        sample_subset["sample_row_index"]
        .to_numpy(int)
    )

    cached_values = signed_cache[
        sample_rows,
        observation_index,
    ].astype(float)

    sample_subset["exact_boundary_distance_m"] = distances
    sample_subset["cached_inside"] = cached_values < 0.0

    return (
        summarize_exact_sample_distances(sample_subset),
        len(sample_subset),
    )

## 7. Trend and ETA helpers

In [ ]:
def fit_distance_trend(time_hours, distances):
    x = np.asarray(time_hours, dtype=float)
    y = np.asarray(distances, dtype=float)

    valid = np.isfinite(x) & np.isfinite(y)

    x = x[valid]
    y = y[valid]

    if len(x) < 2:
        return np.nan, np.nan

    xc = x - x.mean()
    denominator = np.sum(xc ** 2)

    if denominator <= 0:
        return np.nan, np.nan

    slope = np.sum(
        xc * (y - y.mean())
    ) / denominator

    intercept = y.mean() - slope * x.mean()
    predicted = intercept + slope * x

    ss_res = np.sum((y - predicted) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)

    r2 = (
        1.0 - ss_res / ss_tot
        if ss_tot > 0
        else 1.0
    )

    # Negative distance-vs-time slope means approaching.
    return float(-slope), float(r2)


def safe_eta(distance_m, rate_m_per_h):
    if (
        not np.isfinite(distance_m)
        or not np.isfinite(rate_m_per_h)
        or rate_m_per_h <= 0
    ):
        return np.nan

    return float(
        max(distance_m, 0.0)
        / rate_m_per_h
    )


def add_eta(row, name, value):
    row[name] = value
    row[name + "_cap18"] = (
        min(value, ETA_CAP_18H)
        if np.isfinite(value)
        else np.nan
    )
    row[name + "_cap48"] = (
        min(value, ETA_CAP_48H)
        if np.isfinite(value)
        else np.nan
    )

## 8. Per-incident exact-history constructor

In [ ]:
STEP_PREFIX = {
    0: "exact_t0",
    1: "exact_tm1",
    2: "exact_tm2",
    3: "exact_tm3",
}


def build_incident_exact_history(
    incident_base,
    progression_incident,
):
    raw_irwin = str(
        incident_base["_irwin_norm"].iloc[0]
    )

    unit_path = resolve_shard(
        UNIT_ROOT,
        raw_irwin,
        ".parquet",
    )

    sample_path = resolve_shard(
        SAMPLE_ROOT,
        raw_irwin,
        ".parquet",
    )

    trajectory_path = resolve_shard(
        TRAJECTORY_ROOT,
        raw_irwin,
        ".npz",
    )

    units = read_03l_units(unit_path)
    samples = pd.read_parquet(sample_path)

    keep = set(incident_base["unit_uid"])

    units = units[
        units["unit_uid"].isin(keep)
    ].copy()

    samples = samples[
        samples["unit_uid"].isin(keep)
    ].copy()

    weighted_samples = attach_sample_weights(
        units,
        samples,
    )

    with np.load(
        trajectory_path,
        allow_pickle=False,
    ) as cache:
        signed_cache = np.asarray(
            cache["signed"],
            dtype=np.float32,
        )

        cache_times, timestamp_unit = decode_cached_times(
            cache["times_ns"].copy()
        )

    progression_incident = (
        progression_incident
        .sort_values("_time_04a")
        .reset_index(drop=True)
    )

    progression_times = pd.DatetimeIndex(
        progression_incident["_time_04a"]
    )

    if len(progression_times) != len(cache_times):
        raise RuntimeError(
            f"NIROPS/cache count mismatch for {raw_irwin}: "
            f"{len(progression_times)} vs {len(cache_times)}"
        )

    seconds = np.abs(
        (
            pd.Series(progression_times).reset_index(drop=True)
            - pd.Series(cache_times).reset_index(drop=True)
        )
        .dt.total_seconds()
        .to_numpy()
    )

    if np.nanmax(seconds) > 1:
        raise RuntimeError(
            f"NIROPS/cache timestamps differ by >1 second for {raw_irwin}."
        )

    work = incident_base.copy()

    work["prediction_anchor_idx"] = pd.to_numeric(
        work["prediction_anchor_idx"],
        errors="coerce",
    ).astype("Int64")

    anchored = work[
        work["prediction_anchor_idx"].notna()
    ].copy()

    request_frames = []

    for step in range(MAX_HISTORY_OBSERVATIONS):
        request = anchored[
            ["unit_uid", "prediction_anchor_idx"]
        ].copy()

        request["relative_step"] = step
        request["observation_index"] = (
            request["prediction_anchor_idx"].astype(int)
            - step
        )

        request = request[
            request["observation_index"] >= 0
        ]

        request_frames.append(
            request[
                ["unit_uid", "relative_step", "observation_index"]
            ]
        )

    requests = pd.concat(
        request_frames,
        ignore_index=True,
    )

    long_results = []
    n_evaluations = 0

    for observation_index, request_group in requests.groupby(
        "observation_index",
        sort=True,
    ):
        observation_index = int(observation_index)

        fire_geometry = (
            progression_incident.iloc[
                observation_index
            ].geometry
        )

        summary, n_eval = exact_distance_for_request_group(
            requested_units=request_group["unit_uid"].tolist(),
            weighted_samples=weighted_samples,
            signed_cache=signed_cache,
            observation_index=observation_index,
            fire_geometry=fire_geometry,
        )

        n_evaluations += n_eval

        summary = summary.merge(
            request_group[
                ["unit_uid", "relative_step"]
            ],
            on="unit_uid",
            how="inner",
            validate="one_to_one",
        )

        summary["observation_index"] = observation_index
        summary["observation_utc"] = progression_times[
            observation_index
        ]

        long_results.append(summary)

    history_long = pd.concat(
        long_results,
        ignore_index=True,
    )

    result = work[
        [
            "unit_uid",
            "_irwin_norm",
            "IncidentName",
            "Year",
            "_engagement_norm",
            "hold_label",
            "prediction_anchor_idx",
            "prediction_anchor_utc",
            "operational_anchor_clean",
            "prediction_t0_any_inside",
            "local_fire_to_line_distance_geom_m",
            "approach_azimuth_deg",
            "approach_direction_reliable",
            "fireline_orientation_deg",
            "approach_vs_line_angle_deg",
        ]
    ].copy()

    metrics = [
        "exact_mean_distance_m",
        "exact_median_distance_m",
        "exact_p10_distance_m",
        "exact_p90_distance_m",
        "exact_min_distance_m",
        "cached_inside_fraction",
        "sample_points",
        "weight_sum",
        "observation_index",
        "observation_utc",
    ]

    for step, prefix in STEP_PREFIX.items():
        sf = history_long[
            history_long["relative_step"] == step
        ][["unit_uid"] + metrics].copy()

        sf = sf.rename(
            columns={
                c: f"{prefix}_{c}"
                for c in metrics
            }
        )

        result = result.merge(
            sf,
            on="unit_uid",
            how="left",
            validate="one_to_one",
        )

    result["exact_has_previous_observation"] = (
        result["exact_tm1_observation_utc"].notna()
        & result["exact_t0_observation_utc"].notna()
    )

    result["exact_previous_interval_hours"] = (
        (
            pd.to_datetime(
                result["exact_t0_observation_utc"],
                utc=True,
            )
            - pd.to_datetime(
                result["exact_tm1_observation_utc"],
                utc=True,
            )
        )
        .dt.total_seconds()
        / 3600.0
    )

    metric_map = {
        "mean": "exact_mean_distance_m",
        "median": "exact_median_distance_m",
        "p10": "exact_p10_distance_m",
        "min": "exact_min_distance_m",
    }

    for label, metric in metric_map.items():
        result[
            f"exact_last_{label}_approach_rate_m_per_h"
        ] = (
            result[f"exact_tm1_{metric}"]
            - result[f"exact_t0_{metric}"]
        ) / result["exact_previous_interval_hours"]

    trend_rows = []

    for rec in result.to_dict("records"):
        row = {"unit_uid": rec["unit_uid"]}

        t0 = pd.to_datetime(
            rec.get("exact_t0_observation_utc"),
            utc=True,
            errors="coerce",
        )

        t_hours = []
        mean_d = []
        median_d = []
        p10_d = []
        inside = []

        for step, prefix in sorted(
            STEP_PREFIX.items(),
            reverse=True,
        ):
            obs = pd.to_datetime(
                rec.get(f"{prefix}_observation_utc"),
                utc=True,
                errors="coerce",
            )

            if pd.isna(obs) or pd.isna(t0):
                continue

            t_hours.append(
                (obs - t0).total_seconds() / 3600.0
            )

            mean_d.append(
                rec.get(
                    f"{prefix}_exact_mean_distance_m",
                    np.nan,
                )
            )

            median_d.append(
                rec.get(
                    f"{prefix}_exact_median_distance_m",
                    np.nan,
                )
            )

            p10_d.append(
                rec.get(
                    f"{prefix}_exact_p10_distance_m",
                    np.nan,
                )
            )

            inside.append(
                rec.get(
                    f"{prefix}_cached_inside_fraction",
                    np.nan,
                )
            )

        row["exact_history_observation_count"] = len(t_hours)

        row["exact_history_span_hours"] = (
            -min(t_hours)
            if len(t_hours)
            else np.nan
        )

        mean_rate, mean_r2 = fit_distance_trend(
            t_hours,
            mean_d,
        )

        median_rate, median_r2 = fit_distance_trend(
            t_hours,
            median_d,
        )

        p10_rate, p10_r2 = fit_distance_trend(
            t_hours,
            p10_d,
        )

        row["exact_history_mean_approach_rate_m_per_h"] = mean_rate
        row["exact_history_mean_trend_r2"] = mean_r2

        row["exact_history_median_approach_rate_m_per_h"] = median_rate
        row["exact_history_median_trend_r2"] = median_r2

        row["exact_history_p10_approach_rate_m_per_h"] = p10_rate
        row["exact_history_p10_trend_r2"] = p10_r2

        inside_array = np.asarray(inside, dtype=float)

        row["exact_history_any_cached_inside"] = bool(
            np.any(
                inside_array[
                    np.isfinite(inside_array)
                ] > 0
            )
        ) if np.isfinite(inside_array).any() else False

        trend_rows.append(row)

    result = result.merge(
        pd.DataFrame(trend_rows),
        on="unit_uid",
        how="left",
        validate="one_to_one",
    )

    eta_rows = []

    for rec in result.to_dict("records"):
        row = {"unit_uid": rec["unit_uid"]}

        add_eta(
            row,
            "exact_eta_last_mean_h",
            safe_eta(
                rec.get("exact_t0_exact_mean_distance_m", np.nan),
                rec.get("exact_last_mean_approach_rate_m_per_h", np.nan),
            ),
        )

        add_eta(
            row,
            "exact_eta_last_median_h",
            safe_eta(
                rec.get("exact_t0_exact_median_distance_m", np.nan),
                rec.get("exact_last_median_approach_rate_m_per_h", np.nan),
            ),
        )

        add_eta(
            row,
            "exact_eta_last_p10_h",
            safe_eta(
                rec.get("exact_t0_exact_p10_distance_m", np.nan),
                rec.get("exact_last_p10_approach_rate_m_per_h", np.nan),
            ),
        )

        add_eta(
            row,
            "exact_eta_history_mean_h",
            safe_eta(
                rec.get("exact_t0_exact_mean_distance_m", np.nan),
                rec.get("exact_history_mean_approach_rate_m_per_h", np.nan),
            ),
        )

        add_eta(
            row,
            "exact_eta_history_median_h",
            safe_eta(
                rec.get("exact_t0_exact_median_distance_m", np.nan),
                rec.get("exact_history_median_approach_rate_m_per_h", np.nan),
            ),
        )

        eta_rows.append(row)

    result = result.merge(
        pd.DataFrame(eta_rows),
        on="unit_uid",
        how="left",
        validate="one_to_one",
    )

    result["exact_approach_rate_available"] = (
        result["operational_anchor_clean"].fillna(False)
        & result["exact_has_previous_observation"].fillna(False)
        & result["exact_last_mean_approach_rate_m_per_h"].notna()
    )

    result["exact_positive_last_mean_approach"] = (
        result["exact_last_mean_approach_rate_m_per_h"] > 0
    )

    result["exact_eta_last_mean_available"] = (
        result["exact_eta_last_mean_h"].notna()
    )

    metadata = {
        "_irwin_norm": raw_irwin,
        "units": len(result),
        "clean_anchor_units": int(
            result["operational_anchor_clean"].sum()
        ),
        "exact_previous_available_units": int(
            result["exact_has_previous_observation"].sum()
        ),
        "exact_rate_available_units": int(
            result["exact_approach_rate_available"].sum()
        ),
        "exact_positive_last_mean_units": int(
            result["exact_positive_last_mean_approach"].sum()
        ),
        "exact_eta_last_mean_units": int(
            result["exact_eta_last_mean_available"].sum()
        ),
        "sample_observation_evaluations": int(n_evaluations),
        "trajectory_timestamp_integer_unit": timestamp_unit,
    }

    return result, metadata

## 9. One-incident preflight

In [ ]:
TEST_IRWIN = base["_irwin_norm"].iloc[0]

test_incident = base[
    base["_irwin_norm"] == TEST_IRWIN
].copy()

test_key = test_incident["_irwin_04a"].iloc[0]

print("Preflight incident:", TEST_IRWIN)
print("Units:", len(test_incident))

start = time.perf_counter()

test_features, test_metadata = build_incident_exact_history(
    test_incident,
    nirops_groups[test_key],
)

runtime = time.perf_counter() - start

test_metadata["runtime_seconds"] = runtime
test_metadata["sample_observation_eval_per_second"] = (
    test_metadata["sample_observation_evaluations"] / runtime
    if runtime > 0
    else np.nan
)

assert len(test_features) == len(test_incident)
assert test_features["unit_uid"].is_unique

display(pd.DataFrame([test_metadata]))

display(
    test_features[
        [
            "unit_uid",
            "operational_anchor_clean",
            "exact_t0_exact_mean_distance_m",
            "exact_tm1_exact_mean_distance_m",
            "exact_last_mean_approach_rate_m_per_h",
            "exact_history_mean_approach_rate_m_per_h",
            "exact_eta_last_mean_h",
        ]
    ].head(10)
)

print("ONE-INCIDENT PREFLIGHT: PASS")

## 10. Representative runtime benchmark

In [ ]:
inventory = (
    base.groupby(
        ["_irwin_norm", "_irwin_04a"],
        as_index=False,
    )
    .agg(
        units=("unit_uid", "size"),
        clean_units=("operational_anchor_clean", "sum"),
    )
)

inventory["nirops_observations"] = (
    inventory["_irwin_04a"]
    .map({
        key: len(group)
        for key, group in nirops_groups.items()
    })
)

eligible = inventory[
    inventory["nirops_observations"] >= 3
].copy()

small = eligible.sort_values("units").iloc[0]

median_target = eligible["units"].median()
medium = (
    eligible.assign(
        delta=(eligible["units"] - median_target).abs()
    )
    .sort_values("delta")
    .iloc[0]
)

p90_target = eligible["units"].quantile(0.90)
large = (
    eligible.assign(
        delta=(eligible["units"] - p90_target).abs()
    )
    .sort_values("delta")
    .iloc[0]
)

selection = pd.DataFrame([
    {"benchmark_size": "small", **small.to_dict()},
    {"benchmark_size": "medium", **medium.to_dict()},
    {"benchmark_size": "p90_large", **large.to_dict()},
]).drop_duplicates("_irwin_norm")

display(
    selection[
        [
            "benchmark_size",
            "_irwin_norm",
            "units",
            "clean_units",
            "nirops_observations",
        ]
    ]
)

benchmark_rows = []

for rec in selection.to_dict("records"):
    irwin = rec["_irwin_norm"]
    key = rec["_irwin_04a"]

    incident = base[
        base["_irwin_norm"] == irwin
    ].copy()

    print()
    print("=" * 80)
    print(rec["benchmark_size"], irwin, f"units={len(incident):,}")
    print("=" * 80)

    start = time.perf_counter()

    bench_features, bench_meta = build_incident_exact_history(
        incident,
        nirops_groups[key],
    )

    runtime = time.perf_counter() - start
    evaluations = bench_meta["sample_observation_evaluations"]

    benchmark_rows.append({
        "benchmark_size": rec["benchmark_size"],
        "_irwin_norm": irwin,
        "units": len(incident),
        "nirops_observations": rec["nirops_observations"],
        "sample_observation_evaluations": evaluations,
        "runtime_seconds": runtime,
        "evaluations_per_second": (
            evaluations / runtime
            if runtime > 0
            else np.nan
        ),
    })

    del bench_features
    gc.collect()

benchmark = pd.DataFrame(benchmark_rows)

display(benchmark)

benchmark.to_csv(
    OUTPUT_ROOT / "EXACT_DISTANCE_RUNTIME_BENCHMARK_V3.csv",
    index=False,
)

seconds_per_unit = (
    benchmark["runtime_seconds"].sum()
    / benchmark["units"].sum()
)

rough_seconds = seconds_per_unit * len(base)

print()
print(
    "Very rough projected full-production runtime:",
    f"{rough_seconds/60:.1f} minutes",
    f"({rough_seconds/3600:.2f} hours)"
)
print(
    "Treat this only as a rough projection; geometry complexity varies by fire."
)

## 11. Resumable full production

Run this after reviewing the benchmark. Each incident writes one V3 shard. Rerunning the cell reuses completed shards.

In [ ]:
RUN_LOG_PATH = OUTPUT_ROOT / "04A_V3_RUN_LOG.csv"

run_rows = []

groups = list(
    base.groupby("_irwin_norm", sort=False)
)

bar = tqdm(
    groups,
    total=len(groups),
    desc="04A_v3 exact approach incidents",
)

for raw_irwin, incident in bar:
    start = time.perf_counter()

    output_path = SHARD_ROOT / f"{raw_irwin}.parquet"

    try:
        if RESUME and output_path.exists():
            preview = pd.read_parquet(
                output_path,
                columns=[
                    "unit_uid",
                    "exact_approach_rate_available",
                    "exact_eta_last_mean_available",
                ],
            )

            run_rows.append({
                "_irwin_norm": raw_irwin,
                "status": "SKIPPED_COMPLETE",
                "units": len(preview),
                "exact_rate_available_units": int(
                    preview["exact_approach_rate_available"].sum()
                ),
                "exact_eta_last_mean_units": int(
                    preview["exact_eta_last_mean_available"].sum()
                ),
                "runtime_seconds": time.perf_counter() - start,
            })

            bar.set_postfix({
                "incident": str(raw_irwin)[:10],
                "status": "resume",
            })

            continue

        key = incident["_irwin_04a"].iloc[0]

        features, metadata = build_incident_exact_history(
            incident,
            nirops_groups[key],
        )

        features.to_parquet(
            output_path,
            index=False,
        )

        metadata["status"] = "OK"
        metadata["runtime_seconds"] = (
            time.perf_counter() - start
        )

        run_rows.append(metadata)

        bar.set_postfix({
            "incident": str(raw_irwin)[:10],
            "units": metadata["units"],
            "rate": metadata["exact_rate_available_units"],
            "eta": metadata["exact_eta_last_mean_units"],
            "sec": f"{metadata['runtime_seconds']:.1f}",
        })

        del features
        gc.collect()

    except Exception as exc:
        (OUTPUT_ROOT / f"ERROR_{raw_irwin}.txt").write_text(
            traceback.format_exc(),
            encoding="utf-8",
        )

        run_rows.append({
            "_irwin_norm": raw_irwin,
            "status": "ERROR",
            "error": f"{type(exc).__name__}: {exc}",
            "runtime_seconds": time.perf_counter() - start,
        })

        bar.set_postfix({
            "incident": str(raw_irwin)[:10],
            "status": "ERROR",
        })

    pd.DataFrame(run_rows).to_csv(
        RUN_LOG_PATH,
        index=False,
    )

## 12. Completion check and combine

In [ ]:
feature_files = sorted(
    SHARD_ROOT.glob("*.parquet")
)

expected = base["_irwin_norm"].nunique()

print("Completed V3 shards:", len(feature_files), "/", expected)

if len(feature_files) != expected:
    log = pd.read_csv(RUN_LOG_PATH)

    display(
        log[
            log["status"] == "ERROR"
        ]
    )

    raise RuntimeError(
        "04A_v3 incomplete. Fix errors and rerun production; "
        "completed shards will be reused."
    )

exact_approach = pd.concat(
    [
        pd.read_parquet(path)
        for path in tqdm(
            feature_files,
            desc="Combine V3 shards",
        )
    ],
    ignore_index=True,
)

assert exact_approach["unit_uid"].is_unique
assert len(exact_approach) == len(base)

FEATURES_PATH = (
    PROCESSED_ROOT /
    "extended_exact_approach_features_v3.parquet"
)

exact_approach.to_parquet(
    FEATURES_PATH,
    index=False,
)

print("Rows:", f"{len(exact_approach):,}")
print("Saved:", FEATURES_PATH)

## 13. Exact-distance and approach-rate audit

In [ ]:
distance_audit = (
    exact_approach.groupby("_engagement_norm")
    .agg(
        units=("unit_uid", "size"),
        median_exact_t0_mean_m=(
            "exact_t0_exact_mean_distance_m",
            "median",
        ),
        p75_exact_t0_mean_m=(
            "exact_t0_exact_mean_distance_m",
            lambda s: s.quantile(0.75),
        ),
        p90_exact_t0_mean_m=(
            "exact_t0_exact_mean_distance_m",
            lambda s: s.quantile(0.90),
        ),
        p95_exact_t0_mean_m=(
            "exact_t0_exact_mean_distance_m",
            lambda s: s.quantile(0.95),
        ),
        p99_exact_t0_mean_m=(
            "exact_t0_exact_mean_distance_m",
            lambda s: s.quantile(0.99),
        ),
    )
    .reset_index()
)

display(distance_audit)

distance_audit.to_csv(
    OUTPUT_ROOT / "EXACT_T0_DISTANCE_AUDIT_V3.csv",
    index=False,
)


rate_audit = (
    exact_approach.groupby("_engagement_norm")
    .agg(
        units=("unit_uid", "size"),
        clean_anchor_units=("operational_anchor_clean", "sum"),
        previous_available=("exact_has_previous_observation", "sum"),
        rate_available=("exact_approach_rate_available", "sum"),
        positive_last_mean_rate=("exact_positive_last_mean_approach", "sum"),
        median_last_mean_rate_m_per_h=(
            "exact_last_mean_approach_rate_m_per_h",
            "median",
        ),
        p10_last_mean_rate_m_per_h=(
            "exact_last_mean_approach_rate_m_per_h",
            lambda s: s.quantile(0.10),
        ),
        p90_last_mean_rate_m_per_h=(
            "exact_last_mean_approach_rate_m_per_h",
            lambda s: s.quantile(0.90),
        ),
        median_history_mean_rate_m_per_h=(
            "exact_history_mean_approach_rate_m_per_h",
            "median",
        ),
        median_history_mean_r2=(
            "exact_history_mean_trend_r2",
            "median",
        ),
        median_previous_gap_h=(
            "exact_previous_interval_hours",
            "median",
        ),
    )
    .reset_index()
)

rate_audit["rate_available_pct"] = (
    rate_audit["rate_available"]
    / rate_audit["units"]
    * 100
)

rate_audit["positive_last_mean_rate_pct"] = (
    rate_audit["positive_last_mean_rate"]
    / rate_audit["units"]
    * 100
)

display(rate_audit)

rate_audit.to_csv(
    OUTPUT_ROOT / "EXACT_APPROACH_RATE_AUDIT_V3.csv",
    index=False,
)

## 14. Old 251 m-capped versus new exact zero-rate comparison

In [ ]:
if "mean_distance_approach_rate_last_m_per_h" in approach_v2.columns:

    comparison = exact_approach[
        [
            "unit_uid",
            "_engagement_norm",
            "exact_last_mean_approach_rate_m_per_h",
        ]
    ].merge(
        approach_v2[
            [
                "unit_uid",
                "mean_distance_approach_rate_last_m_per_h",
            ]
        ],
        on="unit_uid",
        how="left",
        validate="one_to_one",
    )

    comparison["old_rate_exactly_zero"] = (
        comparison[
            "mean_distance_approach_rate_last_m_per_h"
        ] == 0
    )

    comparison["new_rate_exactly_zero"] = (
        comparison[
            "exact_last_mean_approach_rate_m_per_h"
        ] == 0
    )

    zero_comparison = (
        comparison.groupby("_engagement_norm")
        .agg(
            units=("unit_uid", "size"),
            old_zero=("old_rate_exactly_zero", "sum"),
            new_zero=("new_rate_exactly_zero", "sum"),
        )
        .reset_index()
    )

    zero_comparison["old_zero_pct"] = (
        zero_comparison["old_zero"]
        / zero_comparison["units"]
        * 100
    )

    zero_comparison["new_zero_pct"] = (
        zero_comparison["new_zero"]
        / zero_comparison["units"]
        * 100
    )

else:
    zero_comparison = pd.DataFrame()
    print("Old V2 rate column unavailable; comparison skipped.")

display(zero_comparison)

if len(zero_comparison):
    zero_comparison.to_csv(
        OUTPUT_ROOT /
        "OLD_CAPPED_VS_NEW_EXACT_ZERO_RATE_V3.csv",
        index=False,
    )

## 15. ETA audit

In [ ]:
ETA_COLUMNS = [
    "exact_eta_last_mean_h",
    "exact_eta_last_median_h",
    "exact_eta_last_p10_h",
    "exact_eta_history_mean_h",
    "exact_eta_history_median_h",
]

rows = []

for label, group in exact_approach.groupby("_engagement_norm"):
    for column in ETA_COLUMNS:
        values = group[column].dropna()

        rows.append({
            "_engagement_norm": label,
            "eta_estimator": column,
            "units": len(group),
            "available": len(values),
            "available_pct": len(values) / len(group) * 100,
            "median_h": values.median() if len(values) else np.nan,
            "p10_h": values.quantile(0.10) if len(values) else np.nan,
            "p75_h": values.quantile(0.75) if len(values) else np.nan,
            "p90_h": values.quantile(0.90) if len(values) else np.nan,
            "p95_h": values.quantile(0.95) if len(values) else np.nan,
            "pct_le_3h": (values <= 3).mean()*100 if len(values) else np.nan,
            "pct_le_6h": (values <= 6).mean()*100 if len(values) else np.nan,
            "pct_le_12h": (values <= 12).mean()*100 if len(values) else np.nan,
            "pct_le_18h": (values <= 18).mean()*100 if len(values) else np.nan,
            "pct_le_24h": (values <= 24).mean()*100 if len(values) else np.nan,
            "pct_le_48h": (values <= 48).mean()*100 if len(values) else np.nan,
        })

eta_audit = pd.DataFrame(rows)

display(eta_audit)

eta_audit.to_csv(
    OUTPUT_ROOT / "EXACT_ETA_ESTIMATOR_AUDIT_V3.csv",
    index=False,
)

## 16. Candidate exact-dynamic feature-ready subset

In [ ]:
candidate_exact = exact_approach[
    exact_approach["exact_approach_rate_available"]
].copy()

CANDIDATE_PATH = (
    PROCESSED_ROOT /
    "operational_exact_dynamic_cohort_candidate_v3.parquet"
)

candidate_exact.to_parquet(
    CANDIDATE_PATH,
    index=False,
)

candidate_summary = (
    exact_approach.groupby("_engagement_norm")
    .agg(
        parent_dynamic_units=("unit_uid", "size"),
        clean_anchor_units=("operational_anchor_clean", "sum"),
        exact_rate_ready_units=("exact_approach_rate_available", "sum"),
        positive_last_mean_units=("exact_positive_last_mean_approach", "sum"),
        eta_last_mean_units=("exact_eta_last_mean_available", "sum"),
        incidents=("_irwin_norm", "nunique"),
    )
    .reset_index()
)

candidate_summary["exact_rate_ready_pct"] = (
    candidate_summary["exact_rate_ready_units"]
    / candidate_summary["parent_dynamic_units"]
    * 100
)

candidate_summary["eta_last_mean_pct"] = (
    candidate_summary["eta_last_mean_units"]
    / candidate_summary["parent_dynamic_units"]
    * 100
)

display(candidate_summary)

candidate_summary.to_csv(
    OUTPUT_ROOT /
    "CANDIDATE_EXACT_DYNAMIC_COHORT_SUMMARY_V3.csv",
    index=False,
)

print("Candidate exact-rate rows:", f"{len(candidate_exact):,}")
print("Saved:", CANDIDATE_PATH)

## 17. Final report

In [ ]:
def tomd(frame):
    try:
        return frame.to_markdown(index=False)
    except Exception:
        return frame.to_string(index=False)


report = f"""# 04A_v3 Extended Exact Pre-Challenge Fire-Approach Audit

Build: `{NOTEBOOK_BUILD_ID}`

Generated UTC: {datetime.now(timezone.utc).isoformat()}

## Parent cohort

Dynamic timing-eligible units: **{len(base):,}**

Incidents: **{base["_irwin_norm"].nunique():,}**

04A_v2 prediction anchor and approach direction were reused unchanged.

No external environmental data were downloaded.

No NIROPS rasterization was performed.

## Exact t0 distance distribution

{tomd(distance_audit)}

## Exact observed inter-perimeter approach rates

{tomd(rate_audit)}

## Old capped versus new exact zero-rate comparison

{tomd(zero_comparison) if len(zero_comparison) else "_Comparison unavailable._"}

## ETA candidates

{tomd(eta_audit)}

## Candidate exact-dynamic subset

{tomd(candidate_summary)}

## Runtime benchmark

{tomd(benchmark)}

## Interpretation

The old 251 m far-distance sentinel is no longer used as an approach-distance magnitude.

Exact uncapped Shapely vector boundary-distance magnitude is calculated at up to four observations ending at the accepted operational prediction anchor.

The old 03L signed-distance cache is used only for the existing inside/outside diagnostic.

All rates remain **observed inter-perimeter approach rates**, not instantaneous physical ROS.

A positive approach rate is not required for the feature-ready cohort. Genuine zero or negative rates are retained.

Raw ETA candidates are audited here; no adaptive HRRR horizon is frozen yet.

## Next decision

Choose the ETA formulation only after reviewing:

1. coverage,
2. median and tail behavior,
3. stability across Held and Burned Over,
4. sensitivity to single-interval versus multi-observation trend.

After that, proceed to `04B0_HRRR_access_and_pilot.ipynb`.
"""

REPORT_PATH = (
    OUTPUT_ROOT /
    "EXTENDED_EXACT_APPROACH_REPORT_V3.md"
)

REPORT_PATH.write_text(
    report,
    encoding="utf-8",
)

print(report)
print()
print("Saved:", REPORT_PATH)

## 18. Output inventory

In [ ]:
print("=" * 92)
print("04A_v3 OUTPUTS")
print("=" * 92)

print("Exact approach features:")
print(FEATURES_PATH)

print()
print("Candidate exact-dynamic cohort:")
print(CANDIDATE_PATH)

print()
print("Report:")
print(REPORT_PATH)

print()
for path in sorted(OUTPUT_ROOT.glob("*")):
    if path.is_file():
        print(
            f"{path.name:<68}"
            f"{path.stat().st_size/(1024**2):>10.3f} MB"
        )

print()
print("DONE:", NOTEBOOK_BUILD_ID)